# Validation: 100% Momentum vs 100% Value

This notebook loads MSCI World Momentum and Enhanced Value index data, runs several allocation strategies, and compares performance metrics and plots.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from asset_allocation.backtest import run_backtest
from asset_allocation.config import BacktestConfig
from asset_allocation.data import load_prices
from asset_allocation.metrics import calendar_year_returns, summary
from asset_allocation.plotting import (
    plot_allocation,
    plot_drawdown,
    plot_equity_curve,
    plot_return_distribution,
    plot_summary,
)
from asset_allocation.strategy import (
    buy_and_hold_weights,
    constant_weights,
    periodic_weights,
)

%matplotlib inline

## Load joint price table

In [ ]:
prices = load_prices()
print(f"Observations: {len(prices)}  |  {prices.index[0].date()} -> {prices.index[-1].date()}")
display(prices.head())
display(prices.tail())

ax = prices.div(prices.iloc[0]).plot(figsize=(10, 5), title="Normalized index levels")
ax.set_ylabel("Growth of $1")
ax.grid(True, alpha=0.3)
plt.show()

## Define strategies

- **100% momentum / 100% value**: constant target weights (rebalanced when targets change)
- **50/50 annual**: rebalance to equal weights at each year-end observation
- **50/50 buy-and-hold**: initial 50/50 allocation, no further rebalancing

In [ ]:
strategies = {
    "100% momentum": constant_weights(prices, {"momentum": 1.0}),
    "100% value": constant_weights(prices, {"value": 1.0}),
    "50/50 annual": periodic_weights(prices, {"momentum": 0.5, "value": 0.5}, freq="Y"),
    "50/50 buy-and-hold": buy_and_hold_weights(prices, {"momentum": 0.5, "value": 0.5}),
}

config_default = BacktestConfig()
config_no_friction = BacktestConfig(apply_taxes=False, apply_transaction_costs=False)

## Run backtests (with German taxes & costs)

By default, all holdings are sold on the final evaluation date so exit taxes and transaction costs are included in the reported return.

In [ ]:
results = {
    name: run_backtest(prices, weights, config_default)
    for name, weights in strategies.items()
}

summary_df = pd.concat([summary(r, name) for name, r in results.items()])
summary_df.style.format(
    {
        "total_return": "{:.1%}",
        "cagr": "{:.1%}",
        "annual_volatility": "{:.1%}",
        "sharpe_ratio": "{:.2f}",
        "sortino_ratio": "{:.2f}",
        "calmar_ratio": "{:.2f}",
        "max_drawdown": "{:.1%}",
        "best_month": "{:.1%}",
        "worst_month": "{:.1%}",
        "win_rate": "{:.1%}",
        "annual_turnover": "{:.2f}",
        "time_in_market": "{:.1%}",
        "total_taxes": "{:,.0f}",
        "total_costs": "{:,.0f}",
    },
    na_rep="-",
)

## Sanity check: single-asset buy-and-hold without friction

With taxes and costs disabled, a 100% momentum portfolio should track the momentum index exactly.

In [ ]:
momentum_bh = run_backtest(
    prices,
    constant_weights(prices, {"momentum": 1.0}),
    config_no_friction,
)

normalized_equity = momentum_bh.equity / momentum_bh.equity.iloc[0]
normalized_index = prices["momentum"] / prices["momentum"].iloc[0]
max_tracking_error = (normalized_equity - normalized_index).abs().max()

print(f"Max tracking error vs momentum index: {max_tracking_error:.2e}")
print(f"Number of trades (should be 1): {momentum_bh.num_trades}")

## Plots

In [ ]:
plot_summary(results)
plt.show()

plot_allocation(results["50/50 annual"], title="50/50 Annual Rebalance — Held Weights")
plt.show()

## Calendar year returns (momentum vs value)

In [ ]:
core = {k: results[k] for k in ["100% momentum", "100% value"]}
yearly = pd.DataFrame({name: calendar_year_returns(r) for name, r in core.items()})
yearly.tail(10)